# Private walkthrough rendering comparison

Select a **free T4 GPU**. Stop if payment is required. Free availability varies.

Uploads go to this temporary Google runtime. Keep the notebook private; never commit personal recordings, reconstruction outputs, or download credentials. The app and household schedule are not modified. Download results before deleting the runtime.

Camera poses must be inspected before Gaussian rendering is trusted. A partial component cannot establish a complete home or measured floor plan.


In [ ]:
import pathlib,subprocess,sys
root=pathlib.Path('/content/housekeeping-splat'); root.mkdir(exist_ok=True)
assert subprocess.run(['nvidia-smi'],capture_output=True).returncode == 0, 'Select a free T4 runtime first'
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','colmap','ffmpeg'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
subprocess.run(['uv','venv','--python','3.10',str(root/'venv')],check=True)
python=str(root/'venv/bin/python')
subprocess.run(['uv','pip','install','--python',python,'torch==2.4.1','torchvision==0.19.1','--index-url','https://download.pytorch.org/whl/cu124'],check=True)
subprocess.run(['uv','pip','install','--python',python,'gsplat==1.4.0+pt24cu124','--no-deps','--find-links','https://docs.gsplat.studio/whl/pt24cu124/gsplat/'],check=True)
subprocess.run(['uv','pip','install','--python',python,'numpy==1.26.4','opencv-python-headless==4.10.0.84','pycolmap==3.12.6','nerfstudio==1.1.5','gsplat==1.4.0','torchmetrics==1.6.1','transformers==4.46.3'],check=True)
(root/'align_video.py').write_text('"""Bounded COLMAP trial; no app writes or credentials required.\n\nRun with NumPy, OpenCV and pycolmap 3.12.6 in an isolated environment.\nPersonal inputs and outputs must stay outside Git.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\n\nimport cv2\nimport pycolmap\n\n\ndef extract_frames(video, images, seconds, fps):\n    capture = cv2.VideoCapture(str(video))\n    source_fps = capture.get(cv2.CAP_PROP_FPS)\n    if not capture.isOpened() or source_fps <= 0:\n        raise ValueError("Cannot decode the recording")\n    duration = capture.get(cv2.CAP_PROP_FRAME_COUNT) / source_fps\n    if duration <= 0:\n        capture.release()\n        raise ValueError("The recording has no readable frames")\n    count = 0\n    previews = []\n    targets = {round(i * source_fps / fps): i for i in range(int(min(duration, seconds) * fps))}\n    try:\n        for index in range(max(targets) + 1):\n            if not capture.grab():\n                break\n            if index not in targets:\n                continue\n            ok, frame = capture.retrieve()\n            if not ok:\n                raise ValueError("Unreadable sampled frame")\n            height, width = frame.shape[:2]\n            frame = cv2.resize(frame, (1280, round(height * 1280 / width)))\n            name = f"frame_{count + 1:06d}.jpg"\n            if not cv2.imwrite(str(images / name), frame, [cv2.IMWRITE_JPEG_QUALITY, 95]):\n                raise ValueError("Cannot save sampled frame")\n            if count % (5 * fps) == 0 and len(previews) < 6:\n                import numpy as np\n                ratio = min(480 / frame.shape[1], 270 / frame.shape[0])\n                resized = cv2.resize(frame, (round(frame.shape[1] * ratio), round(frame.shape[0] * ratio)))\n                thumb = np.zeros((270, 480, 3), dtype=np.uint8)\n                y = (270 - resized.shape[0]) // 2\n                x = (480 - resized.shape[1]) // 2\n                thumb[y:y+resized.shape[0], x:x+resized.shape[1]] = resized\n                cv2.putText(thumb, f"{count / fps:g}s", (12, 28), cv2.FONT_HERSHEY_SIMPLEX, .7, (255, 255, 255), 2)\n                previews.append(thumb)\n            count += 1\n    finally:\n        capture.release()\n    if len(previews) == 6:\n        import numpy as np\n        cv2.imwrite(str(images.parent / "contact-sheet.jpg"), np.vstack([np.hstack(previews[i:i+2]) for i in range(0, 6, 2)]))\n    return count, duration\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--video", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--seconds", type=int, default=30, choices=(30, 60, 90))\n    parser.add_argument("--fps", type=int, default=4, choices=(2, 4))\n    parser.add_argument("--matching", default="sequential", choices=("sequential", "exhaustive"))\n    args = parser.parse_args()\n    if args.output.exists():\n        raise ValueError("Use a fresh output folder to preserve earlier results")\n    args.output.mkdir(parents=True)\n    images = args.output / "images"\n    images.mkdir()\n    sparse = args.output / "sparse"\n    sparse.mkdir()\n    report = {"state": "running", "secondsRequested": args.seconds,\n              "fps": args.fps, "width": 1280, "matching": args.matching,\n              "pycolmapVersion": pycolmap.__version__}\n    status = args.output / "report.json"\n    status.write_text(json.dumps(report, indent=2))\n    started = time.monotonic()\n    try:\n        total, duration = extract_frames(args.video, images, args.seconds, args.fps)\n        report.update(extractedFrames=total, sourceDuration=duration, stage="features")\n        status.write_text(json.dumps(report, indent=2))\n        db = str(args.output / "database.db")\n        pycolmap.extract_features(db, str(images), camera_mode=pycolmap.CameraMode.SINGLE,\n                                 camera_model="OPENCV", device=pycolmap.Device.cpu,\n                                 sift_options={"num_threads": 2, "max_image_size": 1280, "max_num_features": 8192})\n        report.update(stage="matching")\n        status.write_text(json.dumps(report, indent=2))\n        if args.matching == "sequential":\n            pycolmap.match_sequential(db, device=pycolmap.Device.cpu,\n                                     sift_options={"num_threads": 2}, matching_options={"overlap": 20, "num_threads": 2})\n        else:\n            pycolmap.match_exhaustive(db, device=pycolmap.Device.cpu, sift_options={"num_threads": 2})\n        report.update(stage="mapping")\n        status.write_text(json.dumps(report, indent=2))\n        pycolmap.incremental_mapping(db, str(images), str(sparse),\n                                    options={"num_threads": 2, "ba_global_function_tolerance": .000001})\n        models = []\n        for folder in sorted(sparse.iterdir()):\n            if not folder.is_dir() or not (folder / "images.bin").exists():\n                continue\n            rec = pycolmap.Reconstruction(str(folder))\n            errors = [float(point.error) for point in rec.points3D.values()]\n            names = sorted(image.name for image in rec.images.values())\n            models.append({"model": folder.name, "registered": len(names),\n                           "points": len(errors), "meanReprojectionError": sum(errors) / len(errors) if errors else None,\n                           "firstFrame": names[0] if names else None, "lastFrame": names[-1] if names else None})\n        models.sort(key=lambda model: model["registered"], reverse=True)\n        best = models[0] if models else None\n        fraction = best["registered"] / total if best and total else 0\n        # A diagnostic gate, not a calibrated accuracy guarantee. Always\n        # inspect trajectory and held-out images before accepting geometry.\n        eligible = bool(best and fraction >= .8 and best["registered"] >= 60\n                        and best["meanReprojectionError"] is not None\n                        and best["meanReprojectionError"] <= 2)\n        report.update(state="complete", extractedFrames=total, components=models,\n                      largestComponentFraction=round(fraction, 4), trainingEligible=eligible,\n                      elapsedSeconds=round(time.monotonic() - started, 2))\n    except Exception as exc:\n        report.update(state="failed", error=type(exc).__name__ + ": " + str(exc),\n                      elapsedSeconds=round(time.monotonic() - started, 2))\n        raise\n    finally:\n        status.write_text(json.dumps(report, indent=2))\n    print(json.dumps(report, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')
(root/'train_component.py').write_text('"""Temporary Linux/T4 rendering comparison using Nerfstudio 1.1.5.\n\nPartial-component training is explicitly labelled and cannot establish a\ncontinuous home reconstruction. This script never calls the cleaning app.\n"""\nimport argparse\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--alignment", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--partial", action="store_true", help="Explicitly permit a labelled partial-component experiment")\n    args = parser.parse_args()\n    report = json.loads((args.alignment / "report.json").read_text())\n    if report["state"] != "complete" or not report.get("components"):\n        raise ValueError("A completed camera reconstruction is required")\n    if not report["trainingEligible"] and not args.partial:\n        raise ValueError("Full-clip alignment failed its gate; no automatic training")\n    best = report["components"][0]\n    if best["registered"] < 20 or best["points"] < 1000 or best["meanReprojectionError"] > 2:\n        raise ValueError("Insufficient geometry even for a partial rendering trial")\n    if args.output.exists():\n        raise ValueError("Use a fresh training folder")\n    import torch\n    if not torch.cuda.is_available():\n        raise RuntimeError("A CUDA GPU is required")\n    args.output.mkdir(parents=True)\n    started = time.monotonic()\n    status = {"state": "running", "stage": "prepare", "gpu": torch.cuda.get_device_name(0),\n              "partial": not report["trainingEligible"], "component": best,\n              "sampledFrames": report["extractedFrames"], "iterations": 3000,\n              "torch": torch.__version__, "python": sys.version,\n              "nerfstudio": importlib.metadata.version("nerfstudio"),\n              "gsplat": importlib.metadata.version("gsplat")}\n    status_path = args.output / "report.json"\n    major, minor = torch.cuda.get_device_capability(0)\n    bin_dir = Path(sys.executable).parent\n    env = {**os.environ, "MAX_JOBS": "2", "TORCH_CUDA_ARCH_LIST": f"{major}.{minor}",\n           "PATH": str(bin_dir) + os.pathsep + os.environ.get("PATH", "")}\n    def run(command, timeout=1800):\n        status_path.write_text(json.dumps(status, indent=2))\n        with (args.output / (status["stage"] + ".log")).open("w") as log:\n            subprocess.run([str(bin_dir / command[0]), *command[1:]], check=True,\n                           env=env, timeout=timeout, stdout=log, stderr=subprocess.STDOUT)\n    try:\n        data = args.output / "data"\n        model = args.alignment / "sparse" / best["model"]\n        run(["ns-process-data", "images", "--data", str(args.alignment / "images"),\n             "--output-dir", str(data), "--skip-colmap", "--colmap-model-path", str(model.resolve()),\n             "--num-downscales", "2"])\n        transforms = json.loads((data / "transforms.json").read_text())\n        if len(transforms["frames"]) != best["registered"]:\n            raise ValueError("Prepared camera count does not match the selected component")\n        status.update(stage="train")\n        run(["ns-train", "splatfacto", "--data", str(data), "--output-dir", str(args.output / "outputs"),\n             "--max-num-iterations", "3000", "--steps-per-save", "1000", "--vis", "tensorboard",\n             "nerfstudio-data", "--downscale-factor", "2"])\n        config = next((args.output / "outputs").rglob("config.yml"))\n        status.update(stage="render", config=str(config))\n        run(["ns-render", "dataset", "--load-config", str(config), "--output-path", str(args.output / "renders"),\n             "--split", "test", "--rendered-output-names", "rgb", "gt-rgb"])\n        status.update(stage="export")\n        run(["ns-export", "gaussian-splat", "--load-config", str(config), "--output-dir", str(args.output / "export")])\n        status.update(state="complete", stage="complete", elapsedSeconds=round(time.monotonic()-started, 2))\n    except Exception as exc:\n        status.update(state="failed", error=type(exc).__name__ + ": " + str(exc))\n        raise\n    finally:\n        status_path.write_text(json.dumps(status, indent=2))\n    print(json.dumps(status, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
from google.colab import files
uploaded=files.upload()
assert len(uploaded)==1, 'Choose exactly one approved MP4 recording'
name,data=next(iter(uploaded.items()))
assert name.lower().endswith('.mp4') and len(data)<=250*1024*1024, 'Use an MP4 below 250 MB'
video=root/'recording.mp4'; video.write_bytes(data)
del uploaded,data


In [ ]:
# Use a fresh output directory for each comparison.
MATCHING='exhaustive' # Slower than 'sequential', but tests all frame pairs.
subprocess.run([python,str(root/'align_video.py'),'--video',str(video),'--output',str(root/'alignment'),'--matching',MATCHING],check=True)
print((root/'alignment/report.json').read_text())


In [ ]:
# Change to True only to intentionally test a labelled partial component.
ALLOW_PARTIAL=False
command=[python,str(root/'train_component.py'),'--alignment',str(root/'alignment'),'--output',str(root/'trial')]
if ALLOW_PARTIAL: command.append('--partial')
subprocess.run(command,check=True)
print((root/'trial/report.json').read_text())


In [ ]:
from IPython.display import display,Image
for path in sorted((root/'trial/renders/test/rgb').glob('*.png'))[:3]: display(Image(filename=str(path)))
files.download(str(root/'trial/report.json'))
for path in (root/'trial/export').glob('*.ply'): files.download(str(path))
